In [1]:
import ssl
import urllib.request

url=("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
file_path="../the-verdict.txt"

# Create an unverified context
context = ssl._create_unverified_context()

# Pass the context to urlopen (Note: urlretrieve doesn't support context directly)
with urllib.request.urlopen(url, context=context) as response, open(file_path, 'wb') as out_file:
    out_file.write(response.read())

In [2]:
from importlib.metadata import version
import tiktoken
print(f"tiktoken version: {version('tiktoken')}")


tiktoken version: 0.12.0


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):

    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids=[]
        self.target_ids=[]

        token_ids=tokenizer.encode(txt)
        for i in range(0, len(token_ids)-max_length-1, stride):
            input_chunk=token_ids[i:i+max_length]
            target_chunk=token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self): return len(self.input_ids)

    def __getitem__(self, idx): return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer=tiktoken.get_encoding('gpt2')
    dataset=GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader=DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

with open(file_path, 'r', encoding='utf-8') as f: raw_txt=f.read()

dataloader=create_dataloader_v1(raw_txt, batch_size=2, max_length=4, stride=1, shuffle=False)
data_iter=iter(dataloader)
first_batch=next(data_iter)
print(f"{[(x.shape, x.dtype) for x in first_batch]=}, \n{first_batch=}")

[(x.shape, x.dtype) for x in first_batch]=[(torch.Size([2, 4]), torch.int64), (torch.Size([2, 4]), torch.int64)], 
first_batch=[tensor([[  40,  367, 2885, 1464],
        [ 367, 2885, 1464, 1807]]), tensor([[ 367, 2885, 1464, 1807],
        [2885, 1464, 1807, 3619]])]


## Using nn.Embedding

In [10]:
# Suppose we have the following 3 training examples, which may represent token IDs in a LLM context
idx=torch.tensor([2,3,1])

# The number of rows in the embedding matrix can be determined by obtaining the largest token ID+1
# If the highest token ID is 3, then we want 4 rows, for the possible token IDs 0, 1, 2, 3
num_idx=max(idx)+1

# The desired embedding dimension is a hyperparameter
out_dim=5

# Let's use the random seed for reproducibility since weights in the embedding layer are initialized with small random values
torch.manual_seed(123)
embedding=torch.nn.Embedding(num_idx, out_dim)

# We can look at the embedding weights
embedding.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  1.5810],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015],
        [ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953]], requires_grad=True)

In [11]:
# We can then use the embedding layers to obtain the vector representation of a training example with ID 1:
embedding(torch.tensor([1]))

tensor([[ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

## Using nn.Linear

In [15]:
# Now we demonstrate that the embedding layer above accomplishes the same as nn.Linear layer on a one-hot encoded representation in PyTorch
# First, let's convert the token IDs into a one-hot representation
onehot=torch.nn.functional.one_hot(idx)
print(f"{onehot=}")

# Now we initialize a Linear layer which carries out a matrix multiplication XW^T
linear=torch.nn.Linear(num_idx, out_dim, bias=False)
print(f"{linear.weight.T.shape=}, \n{linear.weight.T=}")

onehot=tensor([[0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]])
linear.weight.T.shape=torch.Size([4, 5]), 
linear.weight.T=tensor([[ 0.2050,  0.2320, -0.2749,  0.2751,  0.1568],
        [-0.4886,  0.0183, -0.1889,  0.1749,  0.3459],
        [-0.0298,  0.0983, -0.3045, -0.3834, -0.1967],
        [ 0.3526, -0.0473,  0.4153,  0.3858,  0.1060]],
       grad_fn=<PermuteBackward0>)


In [16]:
# Note that the linear layer in PyTorch is also initialized with small random weights; to directly compare it to the Embedding layer above
# we have to use the same small random weights, which is why we reassign them here
linear.weight=torch.nn.Parameter(embedding.weight.T)

# Now we can use the linear layer on the one-hot encoded representation of the inputs
linear(onehot.float())

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]], grad_fn=<MmBackward0>)

In [17]:
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

In [25]:
max_length=4
dataloader=create_dataloader_v1(raw_txt, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter=iter(dataloader)
inputs, targets=next(data_iter)
print(f"Token IDs: {inputs.shape=}, \n{inputs}") # (8,4) data batch consists of 8 text samples with 4 tokens each


vocab_size=50257
output_dim=256
token_embedding_layer=torch.nn.Embedding(vocab_size, output_dim)

token_embeddings=token_embedding_layer(inputs)
print(f"{token_embeddings.shape=}, {token_embeddings.dtype=}")

Token IDs: inputs.shape=torch.Size([8, 4]), 
tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
token_embeddings.shape=torch.Size([8, 4, 256]), token_embeddings.dtype=torch.float32


In [27]:
# Absolute positional embedding approach: we just need to create another embedding layer with the same embedding dimension
context_length=max_length
pos_embedding_layer=torch.nn.Embedding(context_length, output_dim)
pos_embeddings=pos_embedding_layer(torch.arange(context_length)) # 0 to maximum input length -1 (number of tokens in each batch -1)
print(f"{pos_embeddings.shape=}, \n{pos_embeddings}")

pos_embeddings.shape=torch.Size([4, 256]), 
tensor([[-0.8461, -0.4954, -1.3098,  ..., -1.9089, -0.7438, -0.8198],
        [ 0.1931, -0.0072,  1.6352,  ..., -0.3255, -0.0726,  0.7545],
        [-0.3135,  0.6302,  0.9588,  ...,  0.9187, -1.2392, -0.6316],
        [-1.2973,  0.3104,  1.4165,  ...,  0.7389, -0.3831, -0.3234]],
       grad_fn=<EmbeddingBackward0>)


In [28]:
input_embeddings=token_embeddings+pos_embeddings
print(f"{input_embeddings.shape=}")

input_embeddings.shape=torch.Size([8, 4, 256])
